# Selección del Clasificador Emocional

Compara los resultados de fine-tuning de tres arquitecturas sobre EmoEvent con y sin class weights:
- **robertuito** — `pysentimiento/robertuito-base-uncased` (RoBERTa preentrenado en Twitter español)
- **maria** — `PeterPanecillo/PlanTL-GOB-ES-roberta-base-bne-copy` (RoBERTa en corpus español general)
- **beto** — `dccuchile/bert-base-spanish-wwm-cased` (BERT en corpus español general)

## 0. Imports y configuración

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

def save_custom_plot(filename, dpi=300):
    """
    Guarda la figura activa de matplotlib en el directorio de resultados.
    Asegura alta resolución y recorta los márgenes blancos innecesarios.
    """
    output_dir = Path('../data/figures')
    
    # Aseguramos que la extensión .png esté presente
    if not filename.endswith('.png'):
        filename += '.png'
        
    plt.savefig(output_dir / filename, bbox_inches='tight', dpi=dpi)
    print(f"Gráfica guardada: {filename}")

# Estilo global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
EVAL_ROOT   = PROJECT_ROOT / 'eval' / 'emotion_results'
MODELS_ROOT = PROJECT_ROOT / 'models' / 'emotion_classifier'
DATA_DIR    = PROJECT_ROOT / 'data' / 'processed' / 'no_synthetic'

# Orden exacto para mantener la consistencia de colores
EMOTION_ORDER = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'others']
CRITICAL_EMOTIONS = ['fear', 'sadness', 'anger', 'disgust']

# Paleta de colores estándar del proyecto
PALETTE = sns.color_palette('tab10', n_colors=len(EMOTION_ORDER))
EMO_PALETTE = dict(zip(EMOTION_ORDER, PALETTE))
 
# Carpetas disponibles en eval/emotion_results/
RESULT_DIRS = sorted([d for d in EVAL_ROOT.iterdir() if d.is_dir() and (d / 'test_metrics.json').exists()])
print('Carpetas de resultados encontradas:')
for d in RESULT_DIRS:
    print(f'  {d.name}')

In [ ]:
def load_metrics(result_dir: Path) -> dict:
    with open(result_dir / 'test_metrics.json', encoding='utf-8') as f:
        m = json.load(f)
        
    name = result_dir.name
    has_weights = not name.endswith('_no_weights')
    base_name = name.replace('_no_weights', '')
    
    return {
        'carpeta': name,
        'modelo': base_name,
        'class_weights': 'Sí' if has_weights else 'No',
        'accuracy': m['accuracy'],
        'f1_macro': m['f1_macro'],
        **{f'f1_{e}': m['f1_per_class'].get(e, 0.0) for e in EMOTION_ORDER},
    }

# Procesar todos los JSONs y crear el DataFrame principal
rows = [load_metrics(d) for d in RESULT_DIRS]
df_all = pd.DataFrame(rows)

# Ordenar por F1-Macro descendente para que el mejor salga primero
df_all = df_all.sort_values(by='f1_macro', ascending=False).reset_index(drop=True)

## 1. Tabla Comparativa Completa

In [ ]:
# Seleccionamos las columnas de interés
cols_display = [
    'modelo', 'class_weights', 'f1_macro',
    'f1_fear', 'f1_disgust', 'f1_surprise',
    'f1_anger', 'f1_sadness', 'f1_joy', 'f1_others',
    'accuracy',
]

# Creamos una copia para visualización (ya viene ordenada por f1_macro desde la celda anterior)
df_show = df_all[cols_display].copy()

# Renombramos para una presentación formal
rename_dict = {
    'modelo': 'Modelo', 
    'class_weights': 'Class Weights',
    'f1_macro': 'F1 Macro', 
    'accuracy': 'Accuracy',
    **{f'f1_{e}': f'F1 {e.capitalize()}' for e in EMOTION_ORDER},
}
df_show = df_show.rename(columns=rename_dict)

float_cols = [c for c in df_show.columns if c.startswith('F1') or c == 'Accuracy']

# 1. Preparar los datos para el heatmap
# Creamos un índice combinando 'Modelo' y 'Class Weights' para el eje Y
df_heatmap = df_show.copy()
df_heatmap['Configuración'] = df_heatmap['Modelo'] + " (W: " + df_heatmap['Class Weights'].astype(str) + ")"
df_heatmap = df_heatmap.set_index('Configuración')

# Seleccionamos solo las columnas numéricas que ya tenías definidas
float_cols = [c for c in df_heatmap.columns if c.startswith('F1') or c == 'Accuracy']
data_to_plot = df_heatmap[float_cols]

# 2. Configurar el tamaño y estilo de la figura
sns.set_theme(style="white") # Fondo blanco limpio
plt.figure(figsize=(14, 6))

# 3. Crear el heatmap con Seaborn
ax = sns.heatmap(
    data_to_plot,
    annot=True,            # Mostrar los valores numéricos
    fmt=".4f",             # 4 decimales como pedías en tu código
    cmap="RdYlGn",         # Mapa de calor rojo-amarillo-verde
    vmin=0.35,             # Límite inferior (rojo)
    vmax=0.85,             # Límite superior (verde)
    linewidths=2,          # Grosor de la línea blanca separadora (clave para el estilo)
    linecolor='white',     # Color de la línea separadora
    cbar=True,             # Barra de leyenda a la derecha
    annot_kws={"weight": "bold", "size": 11} # Texto en negrita
)

# 4. Personalizar títulos y ejes
ax.set_title('Métricas de evaluación cruzada por Clasificador', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('') # Ocultamos el nombre del eje Y para que quede más limpio
ax.set_xlabel('')

# Rotamos las etiquetas del eje X para que no se superpongan
plt.xticks(rotation=45, ha='right', fontsize=11, fontweight='bold')
plt.yticks(rotation=0, fontsize=11, fontweight='bold')

# Mover el eje X a la parte superior (como en la imagen de referencia, opcional)
# ax.xaxis.tick_top() 

# 5. Guardar la imagen en alta resolución
output_dir = Path('../data/figures')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'heatmap_clasificadores.png'

plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Heatmap guardado correctamente en: {output_path}")


Esta matriz de rendimiento evidencia que Robertuito con ponderación de clases (Fila 0) es la configuración óptima para el sistema. Logra el mejor equilibrio de generalización (F1-Macro de 0.6536) y, lo más importante para la detección del ciberacoso, rescata con éxito la sensibilidad en clases minoritarias críticas como Fear (0.5169) y Disgust (0.6119) sin sacrificar la detección de emociones predominantes como Sadness.

A simple vista, el mapa de calor también confirma nuestra hipótesis sobre el desequilibrio de datos: las versiones sin Class Weights (filas 2 y 5) presentan un menor rendimiento en la clase Fear, confirmando que el aprendizaje sensible al coste era una buena decisión arquitectónica.

## 2. Impacto de Class Weights

In [ ]:
# Construir pares con/sin pesos por modelo base
base_models = df_all['modelo'].unique()

impact_rows = []
for bm in sorted(base_models):
    row_w  = df_all[(df_all['modelo'] == bm) & (df_all['class_weights'] == 'Sí')]
    row_nw = df_all[(df_all['modelo'] == bm) & (df_all['class_weights'] == 'No')]
    if row_w.empty or row_nw.empty:
        continue
    for e in EMOTION_ORDER:
        impact_rows.append({
            'modelo': bm, 'emocion': e,
            'con_pesos': row_w.iloc[0][f'f1_{e}'],
            'sin_pesos': row_nw.iloc[0][f'f1_{e}'],
        })

df_impact = pd.DataFrame(impact_rows)
df_impact['delta'] = df_impact['con_pesos'] - df_impact['sin_pesos']

# Tabla de diferencia crítica
critical_impact = df_impact[df_impact['emocion'].isin(CRITICAL_EMOTIONS)].copy()
pivot_impact = critical_impact.pivot(index='modelo', columns='emocion', values='delta')

display(pivot_impact.style
        .format('{:+.4f}')
        .background_gradient(cmap='RdYlGn', vmin=-0.15, vmax=0.15)
        .set_caption('Diferencia de F1-Score: [Con Pesos] - [Sin Pesos]')
        .set_properties(**{'text-align': 'center', 'border-color': 'lightgrey'}))

In [ ]:
# Barplot agrupado con/sin pesos por clase para cada modelo
n_models = len(sorted(base_models))
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)
if n_models == 1:
    axes = [axes]

x = np.arange(len(EMOTION_ORDER))
width = 0.38

for ax, bm in zip(axes, sorted(base_models)):
    sub = df_impact[df_impact['modelo'] == bm]
    
    # Aseguramos el orden consistente de las emociones
    vals_w  = sub.set_index('emocion').loc[EMOTION_ORDER, 'con_pesos'].values
    vals_nw = sub.set_index('emocion').loc[EMOTION_ORDER, 'sin_pesos'].values
    
    # Colores: Verde para 'Con pesos', Rojo para 'Sin pesos' (intuitivo)
    bars_w  = ax.bar(x - width/2, vals_w,  width, label='Con Class Weights',  color='#2CA02C', alpha=0.85)
    bars_nw = ax.bar(x + width/2, vals_nw, width, label='Sin Class Weights', color='#D62728', alpha=0.85)
    
    for bar, v in zip(bars_w,  vals_w):  ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}', ha='center', fontsize=7.5)
    for bar, v in zip(bars_nw, vals_nw): ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}', ha='center', fontsize=7.5)
    
    ax.set_title(f'Arquitectura: {bm}', fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels([e.capitalize() for e in EMOTION_ORDER], rotation=25, ha='right')
    ax.set_ylim(0, 1.05)
    if ax == axes[0]:
        ax.set_ylabel('F1-score (Test Set)')
    ax.legend(fontsize=9, loc='lower right')
    sns.despine(ax=ax)

plt.suptitle('Comparativa Intra-Modelo: Efecto del Aprendizaje Sensible al Coste', fontsize=14, y=1.05, fontweight='bold')
plt.tight_layout()

save_custom_plot('class_weights_impact')
plt.show()

**El análisis diferencial (Delta F1) confirma la hipótesis planteada en la fase de preprocesamiento:** la inyección de ponderaciones algorítmicas en la función de pérdida (CrossEntropyLoss) es importante para estabilizar la detección de las emociones minoritarias en arquitecturas basadas en RoBERTa. Como se observa en la tabla y los diagramas de barras, tanto MarIA como Robertuito experimentan un incremento drástico en la clase más crítica, Fear (+0.1081 y +0.1031 respectivamente), al activar los Class Weights. Este rescate de la clase minoritaria se logra sin una degradación significativa de las clases mayoritarias, demostrando la eficacia del método.

**La Anomalía Semántica de BETO:** Resulta llamativo el comportamiento divergente de la arquitectura BETO, la cual presenta una penalización en Fear (-0.0373) al aplicar los pesos. Este fenómeno sugiere que el hiperplano de decisión de BETO (entrenado sobre un corpus generalista de Wikipedia) "colapsa" al ser forzado a sobredimensionar clases ruidosas, aumentando los falsos positivos. Sin ponderación, BETO logra un pico en Fear y Disgust, pero lo hace a costa de ocupar el espacio semántico de otras emociones negativas, sufriendo caídas severas en Sadness y Anger (como se vio en la tabla global).

Esta inestabilidad estructural descarta a BETO como modelo candidato, consolidando a las arquitecturas derivadas de RoBERTa (específicamente Robertuito) como las más robustas y predecibles bajo técnicas de balanceo.

## 3. Comparativa entre Modelos (solo versiones CON class weights)

In [ ]:
# Filtrar solo las versiones definitivas (con ponderación)
df_w = df_all[df_all['class_weights'] == 'Sí'].copy().reset_index(drop=True)
print(f"Modelos evaluados: {', '.join(df_w['modelo'].tolist())}")

# Barplot F1 Macro ordenado
df_w_sorted = df_w.sort_values('f1_macro', ascending=True)

# Generar colores graduados (verde para el mejor, rojo para el peor)
colors = [plt.cm.RdYlGn(v) for v in np.linspace(0.4, 0.9, len(df_w_sorted))]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(df_w_sorted['modelo'], df_w_sorted['f1_macro'], color=colors, edgecolor='white')

for bar, v in zip(bars, df_w_sorted['f1_macro']):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, 1.05)
ax.set_xlabel('F1 Macro (Test Set)')
ax.set_title('Rendimiento Global: F1 Macro por Modelo Base', fontweight='bold', pad=15)
sns.despine(left=True)
plt.tight_layout()

save_custom_plot('f1_macro_comparison')
plt.show()

In [ ]:
# Radar chart — F1 por clase para cada modelo
angles = np.linspace(0, 2 * np.pi, len(EMOTION_ORDER), endpoint=False).tolist()
angles += angles[:1]  # Cerrar el polígono para que la línea conecte el final con el principio

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette = sns.color_palette('tab10', len(df_w))

for (_, row), color in zip(df_w.iterrows(), palette):
    values = [row[f'f1_{e}'] for e in EMOTION_ORDER]
    values += values[:1] # Cerrar el polígono de valores
    
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=row['modelo'])
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([e.capitalize() for e in EMOTION_ORDER], fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8'], fontsize=9, color='grey')
ax.set_title('Huella Emocional: F1-Score por Clase', fontsize=14, pad=25, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10, title="Arquitectura")
plt.tight_layout()

save_custom_plot('radar_chart_models')
plt.show()

El análisis comparativo de los tres modelos base, operando bajo la estrategia óptima de ponderación de clases, confirma a Robertuito como el clasificador más solvente del conjunto, alcanzando un F1-Macro de 0.6536. Este liderazgo empírico es teóricamente coherente: al ser un modelo RoBERTa pre-entrenado de forma nativa sobre millones de tweets en español, Robertuito posee una comprensión estructural y semántica superior de la jerga, las abreviaturas y la informalidad propias del microblogging, superando a modelos entrenados sobre corpus formales o periodísticos como BETO o MarIA.

Esta superioridad se ilustra con claridad en el Radar Chart (Huella Emocional). La poligonal de Robertuito (azul) demuestra una consistencia multidimensional, maximizando el área predictiva y "envolviendo" a sus competidores en ejes clave como Sadness, Anger y Fear. Si bien BETO logra un ligero despunte aislado en el vértice de Disgust, lo hace a costa de una contracción severa en el resto del hiperespacio emocional.

En conclusión, la robustez equilibrada de **Robertuito con Class Weights** justifica plenamente su designación como el motor de clasificación de emociones definitivo para la arquitectura del chatbot.

## 4. Análisis de Errores del Mejor Modelo

In [ ]:
# 1. Carga del mejor modelo
best_name = 'robertuito'
best_row = df_w[df_w['modelo'] == best_name].iloc[0]
best_model_path = MODELS_ROOT / best_name

print(f'Modelo seleccionado para Análisis de Errores: {best_name.upper()}')
print(f'  F1 Macro: {best_row["f1_macro"]:.4f}')
print(f'  F1 Fear: {best_row["f1_fear"]:.4f}')
print(f'  F1 Disgust: {best_row["f1_disgust"]:.4f}')
print(f'  F1 Sadness: {best_row["f1_sadness"]:.4f}')

In [ ]:
# 2. Inferencia masiva sobre el Test Set
from src.emotion.emotion_detector import EmotionDetector
print(f'Cargando {best_name}...')
detector = EmotionDetector(model_path=str(best_model_path))

df_test = pd.read_csv(DATA_DIR / 'emoevent_test.csv')
print(f'Ejecutando inferencia sobre {len(df_test)} muestras de test...')

batch_size = 64
texts = df_test['text'].tolist()
all_results = []
for i in range(0, len(texts), batch_size):
    all_results.extend(detector.detect_batch(texts[i:i+batch_size]))

# Extraemos la clase con mayor probabilidad absoluta
y_true = df_test['emotion'].tolist()
y_pred = [max(r.all_scores, key=r.all_scores.get) for r in all_results]
print('Inferencia completada con éxito.\n')

In [ ]:
# Matriz de Confusión Estilizada
cm = confusion_matrix(y_true, y_pred, labels=EMOTION_ORDER, normalize='true')
df_cm = pd.DataFrame(cm, index=[e.capitalize() for e in EMOTION_ORDER], columns=[e.capitalize() for e in EMOTION_ORDER])

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_cm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Proporción (Recall)'})
ax.set_xlabel('Predicho por el modelo', fontsize=12, fontweight='bold', labelpad=15)
ax.set_ylabel('Etiqueta Real Humana', fontsize=12, fontweight='bold', labelpad=15)
ax.set_title(f'Matriz de Confusión Normalizada — {best_name.capitalize()}', fontweight='bold', pad=20)
plt.tight_layout()

save_custom_plot('confusion_matrix_error_analysis')
plt.show()

# Extracción de los Pares de Confusión Críticos
cm_diag_zero = cm.copy()
np.fill_diagonal(cm_diag_zero, 0) # Ignoramos los aciertos
flat_idx = np.argsort(cm_diag_zero.flatten())[::-1][:3] # Top 3 errores

print('Top 3 Falsos Positivos más frecuentes')
confused_pairs = []
for idx in flat_idx:
    real_i, pred_j = divmod(idx, len(EMOTION_ORDER))
    rate = cm_diag_zero[real_i, pred_j]
    print(f'  Real={EMOTION_ORDER[real_i].upper():8s} --> Confundido con={EMOTION_ORDER[pred_j].upper():8s} ({rate:.1%})')
    confused_pairs.append((EMOTION_ORDER[real_i], EMOTION_ORDER[pred_j]))

In [ ]:
# 5 ejemplos concretos de errores para cada par confundido
df_test['pred'] = y_pred

for real_label, pred_label in confused_pairs:
    errors = df_test[
        (df_test['emotion'] == real_label) & (df_test['pred'] == pred_label)
    ][['text', 'emotion', 'pred']].head(5)
    print(f'\nErrores: Real={real_label.upper()} predicho como {pred_label.upper()} ({len(errors)} mostrados)')
    pd.set_option('display.max_colwidth', 110)
    display(errors.reset_index(drop=True))

## 5. Decisión Final

Tras la evaluación del rendimiento de múltiples arquitecturas Transformer y la experimentación con técnicas de aprendizaje sensible al coste, se establecen las siguientes conclusiones definitivas para el módulo de detección emocional:

- **Modelo Definitivo:** Se selecciona la arquitectura Robertuito con ponderación de clases (Class Weights) como el motor principal del sistema. Ha demostrado el mejor equilibrio global (F1-Macro de 0.6536) y una adaptabilidad superior al lenguaje informal de redes sociales.

- **Mitigación del Desequilibrio:** La inyección de pesos en la función de pérdida demostró ser una técnica útil y suficiente. Logró rescatar el rendimiento predictivo en clases minoritarias críticas para el ciberacoso (Fear) sin necesidad de inyectar ruido sintético en el corpus de entrenamiento.

- **Naturaleza de los Errores:** El análisis cualitativo final confirma que los falsos positivos del modelo (ej. confundir Anger con Disgust, o Joy con información neutral de Others) no responden a inestabilidades algorítmicas, sino a la altísima subjetividad y solapamiento semántico del lenguaje humano. El modelo ha aprendido representaciones vectoriales profundas y comete errores en las mismas fronteras lingüísticas donde los anotadores humanos discreparían.